# Laboratorium 2: Analiza danych e-commerce z Apache Spark

**Kurs:** Python w Zastosowaniach Big Data  
**Czas realizacji:** 90 minut 

---

## Cele ksztalcenia

Po wykonaniu cwiczen bedziesz potrafil:

1. Tworzyc i konfigurowac obiekt SparkSession
2. Wczytywac dane i eksplorowac strukture Spark DataFrame
3. Stosowac filtrowanie, selekcje oraz agregacje w PySpark
4. Pisac zapytania Spark SQL na widokach tymczasowych
5. Interpretowac wyniki analizy i formulowac wnioski biznesowe

---

## Kontekst biznesowy

Wcielasz sie w role analityka danych w firmie **SklepPL** — polskiej platformie e-commerce obslugujacej kilka milionow klientow rocznie. Twoj manager przygotowal liste zadan analitycznych, ktorych wyniki posluza do decyzji biznesowych na najblizszy kwartal.

Zbior danych zawiera **500 000 transakcji** z ostatnich 12 miesiecy i zawiera:
- dane zamowien (identyfikator, data, wartosc)
- kategorie produktow i dane geograficzne
- metody platnosci i czas dostawy
- informacje o zwrotach

---

## Instrukcje

- Uruchamiaj komorki po kolei od gory do dolu: `Shift+Enter` lub przycisk `Run`
- NIE modyfikuj komork oznaczonych jako "Konfiguracja" ani "Generowanie danych"
- Wskazowki pojawiaja sie po kliknieciu na komorke z tytulem "Wskazowka"
- Komorki testowe na koncu kazdego cwiczenia sprawdzaja poprawnosc odpowiedzi
- Ocena: **7 cwiczen podstawowych** + **1 bonus**

In [ ]:
# @title Konfiguracja srodowiska -- uruchom te komorke jako pierwsza

!pip install -q pyspark

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, BooleanType
)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (11, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.4

print("Konfiguracja srodowiska zakonczona pomyslnie.")

## Generowanie danych

Ponizej wygenerujemy syntetyczny zbior danych, ktory odwzorowuje rzeczywiste transakcje platformy e-commerce. Dane zawieraja realistyczne wzorce: sezonowosc, roznice regionalne oraz braki danych typowe dla produkcyjnych systemow.

Uruchom ponizsze dwie komorki bez modyfikacji.

In [ ]:
# @title Generowanie danych -- uruchom bez modyfikacji (krok 1/2)

np.random.seed(42)
N = 500_000

CATEGORIES = ["Elektronika", "Odzież", "Dom i Ogród", "Sport", "Ksiązki", "Kosmetyki", "Zabawki", "AGD"]
CITIES = ["Warszawa", "Kraków", "Wrocław", "Poznań", "Gdańsk", "Łódź", "Katowice", "Lublin", "Szczecin", "Bydgoszcz"]
CITY_TO_REGION = {
    "Warszawa": "Mazowieckie", "Kraków": "Małopolskie", "Wrocław": "Dolnośląskie",
    "Poznań": "Wielkopolskie", "Gdańsk": "Pomorskie", "Łódź": "Łódzkie",
    "Katowice": "Śląskie", "Lublin": "Lubelskie", "Szczecin": "Zachodniopomorskie",
    "Bydgoszcz": "Kujawsko-Pomorskie"
}
PAYMENT_METHODS = ["Karta kredytowa", "BLIK", "Przelew", "Pobranie", "PayPal"]

# Srodnia cena produktu (PLN) w kazdej kategorii
BASE_PRICES = {
    "Elektronika": 1200.0, "Odzież": 180.0, "Dom i Ogród": 250.0,
    "Sport": 320.0, "Ksiązki": 45.0, "Kosmetyki": 95.0,
    "Zabawki": 120.0, "AGD": 650.0
}

CATEGORY_WEIGHTS = [0.22, 0.18, 0.14, 0.12, 0.10, 0.09, 0.08, 0.07]
CITY_WEIGHTS    = [0.25, 0.13, 0.10, 0.09, 0.08, 0.08, 0.07, 0.06, 0.07, 0.07]

# Sezonowosc: wzrost sprzedazy w Q4 (Black Friday, Swieta)
start_date = pd.Timestamp("2025-01-01")
day_range = 364
day_indices = np.arange(day_range + 1)
seasonality = 1.0 + 0.8 * np.sin(np.pi * day_indices / 180)
seasonality[300:] += 1.5
seasonality_probs = seasonality / seasonality.sum()

sampled_days   = np.random.choice(day_range + 1, size=N, p=seasonality_probs)
order_dates    = [(start_date + pd.Timedelta(days=int(d))).date().isoformat() for d in sampled_days]
categories_arr = np.random.choice(CATEGORIES, size=N, p=CATEGORY_WEIGHTS)
cities_arr     = np.random.choice(CITIES, size=N, p=CITY_WEIGHTS)
regions_arr    = [CITY_TO_REGION[c] for c in cities_arr]
payment_arr    = np.random.choice(PAYMENT_METHODS, size=N, p=[0.30, 0.35, 0.15, 0.10, 0.10])

unit_prices = np.array([
    max(1.0, np.random.normal(BASE_PRICES[cat], BASE_PRICES[cat] * 0.3))
    for cat in categories_arr
])
quantities = np.random.choice([1, 1, 1, 2, 2, 3, 4, 5], size=N)
discounts  = np.random.choice([0.0, 0.0, 0.0, 0.05, 0.10, 0.15, 0.20, 0.25], size=N)

base_delivery    = np.random.randint(1, 6, size=N)
delivery_penalty = np.where(payment_arr == "Pobranie", np.random.randint(1, 3, size=N), 0)
delivery_days    = base_delivery + delivery_penalty

return_probs = {
    "Elektronika": 0.08, "Odzież": 0.14, "Dom i Ogród": 0.05,
    "Sport": 0.06, "Ksiązki": 0.02, "Kosmetyki": 0.04,
    "Zabawki": 0.05, "AGD": 0.07
}
returned = np.array([np.random.random() < return_probs[cat] for cat in categories_arr])

customer_ids_raw = np.random.randint(10000, 99999, size=N).astype(float)
mask_anon        = np.random.random(size=N) < 0.05
customer_ids     = np.where(mask_anon, np.nan, customer_ids_raw)

orders_pdf = pd.DataFrame({
    "order_id":       [f"ORD-{i+1:07d}" for i in range(N)],
    "customer_id":    customer_ids,
    "order_date":     order_dates,
    "category":       categories_arr,
    "city":           cities_arr,
    "region":         regions_arr,
    "payment_method": payment_arr,
    "quantity":       quantities.astype(int),
    "unit_price":     np.round(unit_prices, 2),
    "discount":       discounts,
    "delivery_days":  delivery_days.astype(int),
    "returned":       returned,
})

print(f"Zbior danych wygenerowany: {len(orders_pdf):,} wierszy, {orders_pdf.shape[1]} kolumn.")

In [ ]:
# @title Tworzenie SparkSession i ladowanie danych -- uruchom bez modyfikacji (krok 2/2)

spark = SparkSession.builder \
    .appName("SklepPL - Analiza Transakcji 2025") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

schema = StructType([
    StructField("order_id",       StringType(),  False),
    StructField("customer_id",    DoubleType(),  True),
    StructField("order_date",     StringType(),  True),
    StructField("category",       StringType(),  True),
    StructField("city",           StringType(),  True),
    StructField("region",         StringType(),  True),
    StructField("payment_method", StringType(),  True),
    StructField("quantity",       IntegerType(), True),
    StructField("unit_price",     DoubleType(),  True),
    StructField("discount",       DoubleType(),  True),
    StructField("delivery_days",  IntegerType(), True),
    StructField("returned",       BooleanType(), True),
])

df = spark.createDataFrame(orders_pdf, schema=schema)

# Wartosc zamowienia po rabacie
df = df.withColumn(
    "order_value",
    F.round(F.col("unit_price") * F.col("quantity") * (1.0 - F.col("discount")), 2)
)

# Kolumna z numerem miesiaca
df = df.withColumn("month", F.month(F.to_date(F.col("order_date"))))

# Kolumna z dniem tygodnia (1=niedziela, 2=poniedzialek, ..., 7=sobota w Spark)
df = df.withColumn("day_of_week", F.dayofweek(F.to_date(F.col("order_date"))))

df.cache()
_ = df.count()  # wymuszamy cache

print(f"SparkSession uruchomiony (Spark {spark.version}).")
print(f"DataFrame zaladowany i zapisany w cache: {df.count():,} wierszy, {len(df.columns)} kolumn.")

---

## Cwiczenie 1: Eksploracja struktury danych

**Poziom trudnosci:** podstawowy  
**Czas:** ok. 10 minut

### Kontekst

Zanim zaczniesz analize, musisz poznac zbior danych. Typowe pierwsze kroki analityka to: sprawdzenie schematu kolumn, ocena skali danych oraz identyfikacja brakow danych.

### Zadania

**a)** Wyswietl schemat DataFrame — nazwy kolumn i ich typy.

**b)** Wyswietl 5 pierwszych wierszy (obserwuj układ danych).

**c)** Oblicz liczbe wszystkich transakcji. Zapisz wynik do zmiennej `total_orders`.

**d)** Policz transakcje z brakujacym `customer_id` (niezalogowani uzytkownicy). Zapisz do zmiennej `anonymous_orders`.

**e)** Wyswietl podstawowe statystyki opisowe dla kolumn `unit_price`, `quantity`, `order_value`, `delivery_days`.

### Pytanie biznesowe

Jaki procent transakcji pochodzi od niezalogowanych uzytkownikow? Czy to wazny segment dla firmy?

In [ ]:
# Cwiczenie 1a: Schemat DataFrame
# TODO: wywolaj metode printSchema() na obiekcie df
...

In [ ]:
# Cwiczenie 1b: Pierwsze wiersze
# TODO: wyswietl 5 pierwszych wierszy uzywajac metody show()
...

In [ ]:
# Cwiczenie 1c i 1d: Liczba transakcji i braki danych

# TODO: policz wszystkie wiersze w df i zapisz do total_orders
total_orders = ...

# TODO: policz wiersze, gdzie customer_id jest wartoscia NULL, zapisz do anonymous_orders
anonymous_orders = ...

print(f"Laczna liczba transakcji: {total_orders:,}")
print(f"Transakcje bez konta klienta (NULL): {anonymous_orders:,} ({100 * anonymous_orders / total_orders:.1f}%)")

In [ ]:
# Cwiczenie 1e: Statystyki opisowe
# TODO: wywolaj describe() z lista kolumn ["unit_price", "quantity", "order_value", "delivery_days"]
#       i wyswietl wynik za pomoca show()
...

In [ ]:
# @title Wskazowka -- Cwiczenie 1
print("""
Wskazowki:
  1a) printSchema() nie przyjmuje zadnych argumentow
  1b) show(n) wyswietla n pierwszych wierszy; show(5, truncate=False) nie obcina tekstow
  1c) count() zwraca liczbe wierszy jako Python int
  1d) Filtrowanie NULL: df.filter(F.col("nazwa_kolumny").isNull()).count()
  1e) describe(["kol1", "kol2"]) + show() wyswietla statystyki tylko wybranych kolumn
""")

In [ ]:
# @title Test automatyczny -- Cwiczenie 1
try:
    assert total_orders == 500_000, f"Oczekiwano 500 000, otrzymano {total_orders}"
    assert 23_000 <= anonymous_orders <= 27_000, (
        f"Oczekiwano okolo 25 000 (5%), otrzymano {anonymous_orders}. "
        f"Sprawdz filtrowanie isNull()."
    )
    print("[OK] Cwiczenie 1 zaliczone.")
    print(f"     Zbior zawiera {total_orders:,} transakcji.")
    print(f"     Uzytkownicy niezalogowani: {anonymous_orders:,} ({100 * anonymous_orders / total_orders:.1f}%)")
    print()
    print("Wniosek biznesowy: Okolo 5% transakcji pochodzi od gosci (bez konta).")
    print("Warto rozwazyc kampanie zachecajaca do rejestracji — pozwolilaby lepiej")
    print("profilowac zachowania zakupowe tej grupy klientow.")
except NameError:
    print("[BLAD] Zmienne total_orders lub anonymous_orders nie zostaly zdefiniowane.")
except AssertionError as e:
    print(f"[NIEPOPRAWNIE] {e}")

---

## Cwiczenie 2: Filtrowanie — identyfikacja zamowien premium

**Poziom trudnosci:** podstawowy  
**Czas:** ok. 12 minut

### Kontekst

Dzial marketingu chce skierowac kampanie lojalnosciowa do klientow, ktorzy zlozyli zamowienia o wartosci powyzej 1 000 PLN. Taka segment to tzw. "premium buyers" — generuja wyzsza marze i sa bardziej lojalni.

### Zadania

**a)** Odfiltruj zamowienia, gdzie `order_value` jest wiekszy niz 1 000 PLN. Zapisz wynik do zmiennej `premium_df`.

**b)** Oblicz liczbe takich zamowien i zapisz do `premium_count`.

**c)** Dla zamowien premium policz ile pochodzi z kazdej kategorii. Posortuj malejaco po liczbie zamowien. Wyswietl wynik.

**d)** Oblicz srednia wartosc zamowienia premium i zapisz do `premium_avg_value` (typ: Python `float`).

### Pytanie biznesowe

Ktora kategoria dominuje wsrod zamowien premium? Czy to zaskakujacy wynik?

In [ ]:
# Cwiczenie 2a i 2b: Filtrowanie zamowien premium

# TODO: odfiltruj wiersze gdzie order_value > 1000 i zapisz do premium_df
premium_df = ...

# TODO: oblicz liczbe wierszy w premium_df i zapisz do premium_count
premium_count = ...

print(f"Zamowienia premium (>1000 PLN): {premium_count:,}")
print(f"Stanowi to {100 * premium_count / total_orders:.1f}% wszystkich transakcji")

In [ ]:
# Cwiczenie 2c: Kategorie wsrod zamowien premium

# TODO: na premium_df zastosuj groupBy("category"), policz wiersze (alias "liczba_zamowien")
#       i posortuj malejaco po liczba_zamowien
premium_by_category = ...

premium_by_category.show()

In [ ]:
# Cwiczenie 2d: Srednia wartosc zamowienia premium

# TODO: oblicz srednia wartosci order_value w premium_df
#       wyciagnij wynik jako Python float i zapisz do premium_avg_value
#       Przykladowy wzorzec: df.agg(F.avg("kolumna")).collect()[0][0]
premium_avg_value = ...

print(f"Srednia wartosc zamowienia premium: {premium_avg_value:,.2f} PLN")

In [ ]:
# @title Wskazowka -- Cwiczenie 2
print("""
Wskazowki:
  2a) Filtrowanie: df.filter(F.col("kolumna") > wartosc)
       Alternatywnie: df.filter(df["kolumna"] > wartosc)
  2b) premium_df.count()
  2c) premium_df.groupBy("category").count().withColumnRenamed("count", "liczba_zamowien").orderBy(F.desc("liczba_zamowien"))
  2d) premium_df.agg(F.avg("order_value").alias("srednia")).collect()[0][0]
       collect() zwraca liste Row, [0][0] wyciaga pierwsza wartosc jako float
""")

In [ ]:
# @title Test automatyczny -- Cwiczenie 2
try:
    assert 60_000 <= premium_count <= 140_000, (
        f"Oczekiwano okolo 85 000-110 000 zamowien premium, otrzymano {premium_count}. "
        f"Sprawdz filtr: order_value > 1000."
    )
    assert premium_avg_value > 1000, (
        f"Srednia wartosci premium powinna byc > 1000 PLN, otrzymano {premium_avg_value:.2f}."
    )
    assert premium_avg_value < 5000, (
        f"Srednia {premium_avg_value:.2f} wydaje sie za wysoka — sprawdz czy filtrujesz poprawna kolumne."
    )
    print(f"[OK] Cwiczenie 2 zaliczone.")
    print(f"     Zamowienia premium: {premium_count:,} ({100 * premium_count / total_orders:.1f}% wszystkich)")
    print(f"     Srednia wartosc: {premium_avg_value:,.2f} PLN")
    print()
    print("Wniosek biznesowy: Segment premium (>1000 PLN) stanowi istotna czesc obrotu.")
    print("Elektronika i AGD dominuja — kampania lojalnosciowa powinna targetowac kupujacych")
    print("w tych kategoriach, oferujac np. przedłuzona gwarancje lub priorytetowa pomoc.")
except NameError as e:
    print(f"[BLAD] Zmienna niezdefiniowana: {e}")
except AssertionError as e:
    print(f"[NIEPOPRAWNIE] {e}")

---

## Cwiczenie 3: Analiza przychodow per kategoria

**Poziom trudnosci:** sredniozaawansowany  
**Czas:** ok. 15 minut

### Kontekst

Zarzad chce wiedziec, ktore kategorie produktow generuja najwiekszy przychod i ile zamowien z nich pochodzi. Wyniki posluza do decyzji o budzecie marketingowym i priorytetach logistycznych na Q1 2026.

### Zadania

**a)** Zgrupuj dane po `category` i oblicz dla kazdej kategorii:
- `laczny_przychod` — suma wartosci wszystkich zamowien (`order_value`)
- `liczba_zamowien` — liczba transakcji
- `srednia_wartosc` — srednia wartosc zamowienia (zaokraglona do 2 miejsc)

Posortuj wynik malejaco po `laczny_przychod`. Zapisz do `category_revenue`.

**b)** Wyciagnij nazwe kategorii z najwyzszym przychodem i zapisz do zmiennej `top_category` (typ: Python `str`).

**c)** Oblicz laczny przychod wszystkich kategorii lacznie (suma kolumny `laczny_przychod` z `category_revenue`) i zapisz do `total_revenue` (typ: Python `float`).

**d)** Uruchom komorke z wizualizacja ponizej, zeby zobaczyc wykres.

### Pytanie biznesowe

Czy dwie najlepsze kategorie lacznie generuja ponad 50% calkowitego przychodu? Jakie decyzje wynikaja z tej koncentracji?

In [ ]:
# Cwiczenie 3a: Przychody per kategoria

# TODO: uzupelnij groupBy i agg
category_revenue = (
    df.groupBy(...)                   # pogrupuj po kategorii
    .agg(
        F.round(F.sum(...), 2).alias("laczny_przychod"),    # suma order_value
        F.count("*").alias("liczba_zamowien"),
        F.round(F.avg(...), 2).alias("srednia_wartosc"),    # srednia order_value
    )
    .orderBy(F.desc("laczny_przychod"))
)

category_revenue.show()

In [ ]:
# Cwiczenie 3b i 3c: Top kategoria i laczny przychod

# TODO: wyciagnij nazwe kategorii z pierwszego wiersza category_revenue
#       Wzorzec: df.first()["nazwa_kolumny"]
top_category = ...

# TODO: oblicz sume kolumny "laczny_przychod" z category_revenue jako Python float
total_revenue = ...

print(f"Kategoria z najwyzszym przychodem: {top_category}")
print(f"Laczny przychod platformy: {total_revenue:,.0f} PLN")

In [ ]:
# @title Wskazowka -- Cwiczenie 3
print("""
Wskazowki:
  3a) df.groupBy("category").agg(F.round(F.sum("order_value"), 2).alias("laczny_przychod"), ...)
  3b) category_revenue.first()["laczny_przychod"] lub category_revenue.first()[0]
       Do nazwy kategorii: category_revenue.first()["category"]
  3c) category_revenue.agg(F.sum("laczny_przychod")).collect()[0][0]
""")

In [ ]:
# @title Test automatyczny -- Cwiczenie 3
try:
    assert category_revenue.count() == 8, (
        f"Oczekiwano 8 kategorii, otrzymano {category_revenue.count()}."
    )
    assert top_category == "Elektronika", (
        f"Oczekiwano 'Elektronika' jako top kategoria, otrzymano '{top_category}'."
    )
    assert total_revenue > 100_000_000, (
        f"Laczny przychod {total_revenue:,.0f} PLN wydaje sie za niski."
    )
    print("[OK] Cwiczenie 3 zaliczone.")
    print(f"     Laczny przychod: {total_revenue:,.0f} PLN")
    print(f"     Kategoria nr 1: {top_category}")
    print()
    print("Wniosek biznesowy: Elektronika dominuje przychodowo mimo sredniej popularnosci")
    print("(22% zamowien). Wysoka cena jednostkowa sprawia, ze ta kategoria powinna byc")
    print("priorytetem w sprzedazy wiazanej i programach ubezpieczenia urzadzen.")
except NameError as e:
    print(f"[BLAD] Zmienna niezdefiniowana: {e}")
except AssertionError as e:
    print(f"[NIEPOPRAWNIE] {e}")

In [ ]:
# Wizualizacja wynikow Cwiczenia 3 -- uruchom po ukonczeniu cwiczenia

cr_pd = category_revenue.toPandas().sort_values("laczny_przychod")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Laczny przychod per kategoria
bars = axes[0].barh(cr_pd["category"], cr_pd["laczny_przychod"] / 1e6, color="#2d866b")
axes[0].set_xlabel("Przychod (mln PLN)")
axes[0].set_title("Laczny przychod per kategoria")
for bar, val in zip(bars, cr_pd["laczny_przychod"] / 1e6):
    axes[0].text(val + 0.5, bar.get_y() + bar.get_height() / 2,
                 f"{val:.1f}", va="center", fontsize=9)

# Srednia wartosc zamowienia per kategoria
cr_sorted = cr_pd.sort_values("srednia_wartosc")
axes[1].barh(cr_sorted["category"], cr_sorted["srednia_wartosc"], color="#1a3a5c")
axes[1].set_xlabel("Srednia wartosc zamowienia (PLN)")
axes[1].set_title("Srednia wartosc zamowienia per kategoria")

plt.tight_layout()
plt.show()
print("Interperetacja: Na lewym wykresie widac bezwzgledny wklad kategorii w przychod.")
print("Na prawym — ktore kategorie maja najwyzszy ticket (srednie zamowienie). Elektronika")
print("i AGD wyrozniaja sie na obu wykresach, co potwierdza ich kluczowa role dla marzy.")

---

## Cwiczenie 4: Regionalna analiza sprzedazy

**Poziom trudnosci:** sredniozaawansowany

### Kontekst

Dzial logistyki chce lepiej rozdzielic budzetna marketing i dostosowac oferte do poszczegolnych regionow. Potrzebujesz raportu pokazujacego, jak sprzedaz rozklada sie geograficznie: ile zamowien pochodzi z kazdego regionu, jaki jest sredni czas dostawy i jaki procent zamowien jest zwracany.

### Zadania

a) Pogrupuj dane po kolumnie `region` i oblicz nastepujace metryki dla kazdego regionu:
   - liczba zamowien (`liczba_zamowien`)
   - laczny przychod (`laczny_przychod`)
   - sredni czas dostawy w dniach (`sredni_czas_dostawy`)
   - procent zwroconych zamowien (`procent_zwrotow`) — wyraze jako liczbe od 0 do 100

   Wynik posortuj malejaco po `laczny_przychod` i zapisz do zmiennej `region_stats`.

b) Wyswietl wynik. Ilu unikalnych regionow oczekujesz?

c) Zapisz nazwe regionu z najwyzszym lacznym przychodem do zmiennej `top_region`.

d) Zapisz sredni procent zwrotow dla wszystkich regionow lacznei do zmiennej `avg_return_rate` (jeden wynik, nie per region).

**Podpowiedz do procentu zwrotow:**
Kolumna `returned` jest typu boolean. W Sparku mozna uzyc: `F.avg(F.col("returned").cast("double")) * 100`

**Oczekiwane wyniki:**
- Liczba regionow: 10
- Najwyzszy przychod: Mazowieckie (Warszawa — duza populacja i zamoznosc)
- Sredni procent zwrotow: okolo 6-8%

In [ ]:
# Cwiczenie 4a: Regionalna analiza sprzedazy

# TODO: uzupelnij agregacje ponizej (zastap ... poprawnymi wyrazeniami)
region_stats = (
    df.groupBy("region")
    .agg(
        F.count("order_id").alias("liczba_zamowien"),
        F.round(F.sum("order_value"), 2).alias("laczny_przychod"),
        F.round(F.avg("delivery_days"), 2).alias("sredni_czas_dostawy"),
        F.round(F.avg(F.col("returned").cast("double")) * 100, 2).alias("procent_zwrotow"),
    )
    .orderBy(F.desc("laczny_przychod"))
)

# Cwiczenie 4b: Wyswietl wynik
# TODO: wyswietl region_stats
...

# Cwiczenie 4c: Najlepszy region
# TODO: pobierz nazwe regionu z najwyzszym przychodem i zapisz do top_region
top_region = ...

# Cwiczenie 4d: Sredni procent zwrotow (ogolny)
# TODO: oblicz sredni procent zwrotow dla calego zbioru (nie per region) i zapisz do avg_return_rate
avg_return_rate = ...

print(f"Region z najwyzszym przychodem: {top_region}")
print(f"Sredni procent zwrotow: {avg_return_rate:.2f}%")

In [ ]:
# @title Wskazowka do Cwiczenia 4
print("""Wskazowki:
- groupBy("region").agg(...) -- ten sam schemat co w cwiczeniu 3
- Do obliczenia procentu zwrotow: F.avg(F.col("returned").cast("double")) * 100
  Kolumna boolean jest konwertowanna na 0/1, wiec avg() daje udzial, a *100 daje procent
- top_region: region_stats.first()["region"] -- first() zwraca pierwszy wiersz (po posortowaniu to najlepszy)
- avg_return_rate: uzyj df.select(F.avg(...).alias("avg")).collect()[0]["avg"]
  lub df.agg(F.avg(...)).collect()[0][0]
""")

In [ ]:
# @title Test automatyczny -- Cwiczenie 4
try:
    assert region_stats.count() == 10, (
        f"Oczekiwano 10 regionow, otrzymano {region_stats.count()}. "
        "Sprawdz grupowanie po kolumnie 'region'."
    )
    assert top_region == "Mazowieckie", (
        f"Oczekiwano Mazowieckie, otrzymano '{top_region}'. "
        "Sprawdz sortowanie i pobieranie pierwszego wiersza."
    )
    assert 5.0 <= avg_return_rate <= 10.0, (
        f"Oczekiwano sredniego procentu zwrotow miedzy 5 a 10%, otrzymano {avg_return_rate:.2f}%. "
        "Sprawdz obliczenia -- pamietaj o *100."
    )
    print("POPRAWNIE. Analiza regionalna poprawna.")
    print(f"Najlepszy region: {top_region}, sredni zwrot: {avg_return_rate:.2f}%")
    print("Wniosek biznesowy: Mazowieckie to priorytetowy region dla kampanii marketingowych.")
except NameError as e:
    print(f"BLAD: Zmienna nie zostala zdefiniowana -- {e}")
except AssertionError as e:
    print(f"NIEPOPRAWNIE: {e}")

In [ ]:
# Wizualizacja wynikow Cwiczenia 4 -- uruchom po ukonczeniu cwiczenia

rp = region_stats.toPandas().sort_values("laczny_przychod")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].barh(rp["region"], rp["laczny_przychod"] / 1e6, color="#2d6a8f")
axes[0].set_xlabel("Laczny przychod (mln PLN)")
axes[0].set_title("Przychod per region")

axes[1].barh(rp["region"], rp["sredni_czas_dostawy"], color="#8f2d2d")
axes[1].set_xlabel("Sredni czas dostawy (dni)")
axes[1].set_title("Sredni czas dostawy per region")

plt.tight_layout()
plt.show()
print("Interpretacja: Mazowieckie dominuje pod wzgledem przychodow ze wzgledu na duze zaludnienie")
print("i wysoka sile nabywcza. Roznice w czasie dostawy miedzy regionami sa nieznaczne,")
print("co sugeruje, ze logistyka dziala rowno w calym kraju.")

---

## Cwiczenie 5: Spark SQL — sezonowosc sprzedazy

**Poziom trudnosci:** sredniozaawansowany

### Kontekst

Jednym z kluczowych narzedzi w Apache Spark jest mozliwosc pisania zapytan SQL bezposrednio na DataFrame. Pozwala to analitykom zaznajomionym z SQL pracowac z duzymi danymi bez nauki nowej skladni. W tym cwiczeniu uzyjesz Spark SQL do zbadania sezonowosci sprzedazy.

Rozumienie sezonowosci pomaga planowac zapasy, kampanie reklamowe i zatrudnienie w magazynach.

### Przygotowanie

Najpierw zarejestruj DataFrame jako tymczasowy widok SQL, na ktorym bedziesz wykonywac zapytania.

```python
df.createOrReplaceTempView("transakcje")
```

### Zadania

a) Zarejestruj DataFrame `df` jako widok o nazwie `transakcje`.

b) Napisz zapytanie SQL, ktore zwroci laczny przychod i liczbe zamowien dla kazdego miesiaca, posortowane rosnaco po miesiacu. Zapisz wynik do zmiennej `monthly_sales`.

   Kolumna miesiac: uzyj `MONTH(order_date)` lub wczesniej obliczonej kolumny `month`.

c) Napisz zapytanie SQL, ktore zwroci laczny przychod dla kazdego dnia tygodnia (1=poniedzialek, 7=niedziela). Uzyj `DAYOFWEEK(order_date)` lub kolumny `day_of_week`. Zapisz wynik do zmiennej `weekday_sales`.

d) Zapisz numer miesiaca z najwyzszym przychodem do zmiennej `peak_month` (jako Python int).

e) Zapisz numer dnia tygodnia z najwyzszym przychodem do zmiennej `peak_weekday` (jako Python int).

**Oczekiwane wyniki:**
- Najlepszy miesiac: 11 lub 12 (Black Friday w listopadzie, efekt swiateczny w grudniu)
- Dane naleza do platformy e-commerce, wiec weekendy moga miec wyzszą aktywnosc klientow

In [ ]:
# Cwiczenie 5a: Rejestracja widoku SQL
# TODO: zarejestruj df jako widok "transakcje" uzywajac createOrReplaceTempView
...

# Cwiczenie 5b: Sprzedaz per miesiac
# TODO: napisz zapytanie SQL i zapisz wynik do monthly_sales
monthly_sales = spark.sql("""
    -- TODO: SELECT month, SUM(order_value) AS ..., COUNT(*) AS ...
    -- FROM transakcje
    -- GROUP BY month
    -- ORDER BY month
""")
monthly_sales.show()

# Cwiczenie 5c: Sprzedaz per dzien tygodnia
# TODO: napisz zapytanie SQL i zapisz wynik do weekday_sales
weekday_sales = spark.sql("""
    -- TODO: podobne zapytanie ale grupuj po day_of_week
""")
weekday_sales.show()

# Cwiczenie 5d: Miesiac z najwyzszym przychodem
# TODO: pobierz numer miesiaca z najwyzszym lacznym przychodem i zapisz do peak_month (jako int)
peak_month = ...

# Cwiczenie 5e: Dzien tygodnia z najwyzszym przychodem
# TODO: pobierz numer dnia tygodnia z najwyzszym przychodem i zapisz do peak_weekday (jako int)
peak_weekday = ...

WEEKDAY_NAMES = {1: "Poniedzialek", 2: "Wtorek", 3: "Sroda", 4: "Czwartek",
                 5: "Piatek", 6: "Sobota", 7: "Niedziela"}
print(f"Szczytowy miesiac sprzedazy: {peak_month}")
print(f"Szczytowy dzien tygodnia:    {WEEKDAY_NAMES.get(peak_weekday, peak_weekday)}")

In [ ]:
# @title Wskazowka do Cwiczenia 5
print("""Wskazowki:
Cwiczenie 5a:
  df.createOrReplaceTempView("transakcje")

Cwiczenie 5b -- przykladowe zapytanie:
  SELECT month, SUM(order_value) AS laczny_przychod, COUNT(*) AS liczba_zamowien
  FROM transakcje
  GROUP BY month
  ORDER BY month

Cwiczenie 5d -- pobieranie najlepszego miesiaca:
  peak_row = monthly_sales.orderBy(F.desc("laczny_przychod")).first()
  peak_month = int(peak_row["month"])

Cwiczenie 5e -- analogicznie dla dnia tygodnia.
Uwaga: DAYOFWEEK() w Sparku zwraca 1=niedziela, 7=sobota (standard SQL).
Kolumna day_of_week w naszym zbiorze uzyta jest pandas.Timestamp.dayofweek:
  0=poniedzialek, 6=niedziela (standard Pythona).
Wybierz jedna z dostepnych kolumn i badzspojny.
""")

In [ ]:
# @title Test automatyczny -- Cwiczenie 5
try:
    assert monthly_sales.count() == 12, (
        f"Oczekiwano 12 miesiecy, otrzymano {monthly_sales.count()}. "
        "Sprawdz zapytanie SQL i grupowanie po miesiacu."
    )
    assert peak_month in (11, 12), (
        f"Oczekiwano miesiaca 11 lub 12, otrzymano {peak_month}. "
        "Sprawdz logike sezonowosci: Black Friday (listopad) i swietaBozego Narodzenia (grudzien) "
        "powinny dawac najwyzsze przychody."
    )
    assert weekday_sales.count() == 7, (
        f"Oczekiwano 7 dni tygodnia, otrzymano {weekday_sales.count()}."
    )
    print("POPRAWNIE. Analiza sezonowosci poprawna.")
    MONTH_NAMES = {11: "listopad", 12: "grudzien"}
    print(f"Szczytowy miesiac sprzedazy: {peak_month} ({MONTH_NAMES[peak_month]})")
    print("Wniosek biznesowy: Kampanie reklamowe i zapasy magazynowe nalezy planowac")
    print("juz od pazdziernika, aby byc gotowym na wzmozony ruch w Q4.")
except NameError as e:
    print(f"BLAD: Zmienna nie zostala zdefiniowana -- {e}")
except AssertionError as e:
    print(f"NIEPOPRAWNIE: {e}")

In [ ]:
# Wizualizacja wynikow Cwiczenia 5 -- uruchom po ukonczeniu cwiczenia

mp = monthly_sales.toPandas().sort_values("month")
MONTH_SHORT = ["Sty", "Lut", "Mar", "Kwi", "Maj", "Cze",
               "Lip", "Sie", "Wrz", "Paz", "Lis", "Gru"]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

colors_month = ["#d9534f" if m in (11, 12) else "#5b9bd5" for m in mp["month"]]
axes[0].bar(mp["month"], mp["laczny_przychod"] / 1e6, color=colors_month)
axes[0].set_xticks(range(1, 13))
axes[0].set_xticklabels(MONTH_SHORT)
axes[0].set_ylabel("Przychod (mln PLN)")
axes[0].set_title("Laczny przychod per miesiac (czerwony = Q4)")
axes[0].axvline(x=10.5, color="gray", linestyle="--", linewidth=1, label="Poczatek Q4")
axes[0].legend()

wp = weekday_sales.toPandas().sort_values("day_of_week")
WEEKDAY_SHORT = ["Pon", "Wto", "Sro", "Czw", "Pia", "Sob", "Nie"]
axes[1].bar(range(7), wp.sort_values("day_of_week")["laczny_przychod"] / 1e6, color="#5b9bd5")
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(WEEKDAY_SHORT)
axes[1].set_ylabel("Przychod (mln PLN)")
axes[1].set_title("Laczny przychod per dzien tygodnia")

plt.tight_layout()
plt.show()

---

## Cwiczenie 6: Analiza metod platnosci i zwrotow

**Poziom trudnosci:** sredniozaawansowany

### Kontekst

Dzial operacyjny analizuje, czy metoda platnosci ma wplyw na czas dostawy i wskaznik zwrotow. Hipoteza: zamowienia oplacone przy odbiorze (Pobranie) sa dostarczane pozniej, bo wymagaja rozliczenia z kuriern. Dzial finansowy natomiast chce wiedziec, ktora kategoria produktow ma najwyzszy wskaznik zwrotow — to bezposrednio wplywa na koszty logistyki.

### Zadania

a) Pogrupuj dane po `payment_method` i oblicz dla kazdej metody:
   - liczbe zamowien (`liczba_zamowien`)
   - sredni czas dostawy w dniach (`sredni_czas_dostawy`)
   - procent zwroconych zamowien (`procent_zwrotow`)

   Posortuj wynik malejaco po `sredni_czas_dostawy`. Zapisz do zmiennej `payment_analysis`.

b) Pobierz sredni czas dostawy dla metody "Pobranie" i zapisz do zmiennej `cod_avg_delivery` (float).

c) Pobierz sredni czas dostawy dla metody "BLIK" i zapisz do zmiennej `blik_avg_delivery` (float).

d) Pogrupuj dane po `category` i oblicz wskaznik zwrotow (procent). Posortuj malejaco.
   Zapisz do zmiennej `return_by_category`.

e) Zapisz nazwe kategorii z najwyzszym wskaznikiem zwrotow do zmiennej `highest_return_category`.

**Oczekiwane wyniki:**
- Pobranie powinno miec dluzszy sredni czas dostawy niz BLIK (o 1-2 dni)
- Kategoria z najwyzszym wskaznikiem zwrotow: Odzież (kupujemy, mierzymy, oddajemy)

In [ ]:
# Cwiczenie 6a: Analiza per metoda platnosci
# TODO: uzupelnij groupBy i agg
payment_analysis = (
    df.groupBy("payment_method")
    .agg(
        ...  # liczba_zamowien
        ...  # sredni_czas_dostawy
        ...  # procent_zwrotow
    )
    .orderBy(F.desc("sredni_czas_dostawy"))
)
payment_analysis.show()

# Cwiczenie 6b: Sredni czas dostawy dla Pobranie
# TODO: przefiltruj payment_analysis i pobierz wartosc sredni_czas_dostawy
cod_avg_delivery = ...

# Cwiczenie 6c: Sredni czas dostawy dla BLIK
# TODO: analogicznie dla BLIK
blik_avg_delivery = ...

print(f"Sredni czas dostawy -- Pobranie: {cod_avg_delivery:.2f} dni")
print(f"Sredni czas dostawy -- BLIK:     {blik_avg_delivery:.2f} dni")

# Cwiczenie 6d: Wskaznik zwrotow per kategoria
# TODO: groupBy category, oblicz procent_zwrotow, posortuj malejaco
return_by_category = ...
return_by_category.show()

# Cwiczenie 6e: Kategoria z najwyzszym wskaznikiem zwrotow
# TODO: pobierz nazwe kategorii z najwyzszym procentem zwrotow
highest_return_category = ...
print(f"Kategoria z najwyzszym wskaznikiem zwrotow: {highest_return_category}")

In [ ]:
# @title Wskazowka do Cwiczenia 6
print("""Wskazowki:
Cwiczenie 6a:
  .agg(
      F.count("order_id").alias("liczba_zamowien"),
      F.round(F.avg("delivery_days"), 2).alias("sredni_czas_dostawy"),
      F.round(F.avg(F.col("returned").cast("double")) * 100, 2).alias("procent_zwrotow"),
  )

Cwiczenie 6b i 6c -- pobieranie wartosci z przefiltrowanego DataFrame:
  cod_avg_delivery = (
      payment_analysis
      .filter(F.col("payment_method") == "Pobranie")
      .collect()[0]["sredni_czas_dostawy"]
  )
  -- lub: .select("sredni_czas_dostawy").collect()[0][0]

Cwiczenie 6d:
  return_by_category = (
      df.groupBy("category")
      .agg(F.round(F.avg(F.col("returned").cast("double")) * 100, 2).alias("procent_zwrotow"))
      .orderBy(F.desc("procent_zwrotow"))
  )

Cwiczenie 6e:
  highest_return_category = return_by_category.first()["category"]
""")

In [ ]:
# @title Test automatyczny -- Cwiczenie 6
try:
    assert payment_analysis.count() == 5, (
        f"Oczekiwano 5 metod platnosci, otrzymano {payment_analysis.count()}."
    )
    assert cod_avg_delivery > blik_avg_delivery, (
        f"Oczekiwano, ze Pobranie ({cod_avg_delivery:.2f}) bedzie wolniejsze niz BLIK ({blik_avg_delivery:.2f}). "
        "Sprawdz filtrowanie lub obliczenia sredniej."
    )
    assert return_by_category.count() == 8, (
        f"Oczekiwano 8 kategorii, otrzymano {return_by_category.count()}."
    )
    assert highest_return_category == "Odzież", (
        f"Oczekiwano kategorii 'Odzież', otrzymano '{highest_return_category}'. "
        "Sprawdz sortowanie malejace po wskazniku zwrotow."
    )
    print("POPRAWNIE. Analiza metod platnosci i zwrotow poprawna.")
    print(f"Pobranie vs BLIK: {cod_avg_delivery:.2f} vs {blik_avg_delivery:.2f} dni (roznica: {cod_avg_delivery - blik_avg_delivery:.2f} dni)")
    print(f"Kategoria z najwyzszym wskaznikiem zwrotow: {highest_return_category}")
    print("Wniosek biznesowy: Oplata przy odbiorze wydluza dostawe. Zachecanie klientow")
    print("do platnosci elektronicznych (BLIK, karta) moze skrocic czas realizacji.")
    print("Odzież wymaga specjalnej polityki zwrotow i dodatkowej przestrzeni magazynowej.")
except NameError as e:
    print(f"BLAD: Zmienna nie zostala zdefiniowana -- {e}")
except AssertionError as e:
    print(f"NIEPOPRAWNIE: {e}")

---

## Cwiczenie 7 (Bonus): Segmentacja klientow wedlug wartosci

**Poziom trudnosci:** zaawansowany

### Kontekst

Dzial marketingu chce wdrozyc program lojalnosciowy oparty na wartosci klienta. Segmentacja klientow pozwala dostosowac komunikacje: klienci VIP otrzymuja ekskluzywne oferty, regularne — program punktowy, a okazjonalni — zachetedo ponownego zakupu.

W tym cwiczeniu wdrozysz uproszczona wersje analizy wartosci klienta (ang. Customer Value Segmentation).

### Definicja segmentow

| Segment       | Kryterium                                           |
|---------------|-----------------------------------------------------|
| VIP           | laczna wartosc zamowien powyzej 5000 PLN            |
| Regularny     | laczna wartosc zamowien od 1000 do 5000 PLN         |
| Okazjonalny   | laczna wartosc zamowien ponizej 1000 PLN            |

### Zadania

a) Przefiltruj DataFrame tak, aby usunac wiersze z brakujacym `customer_id` (niezalogowani klienci nie maja profilu i nie mozna ich segmentowac). Wynik zapisz do zmiennej `df_logged`.

b) Pogrupuj `df_logged` po `customer_id` i oblicz dla kazdego klienta:
   - laczna wartosc zamowien (`total_spent`)
   - liczba zamowien (`order_count`)
   - srednia wartosc zamowienia (`avg_order_value`)

   Zapisz wynik do zmiennej `customer_stats`.

c) Dodaj do `customer_stats` kolumne `segment` uzywajac `F.when(...).when(...).otherwise(...)`:
   - "VIP" gdy `total_spent > 5000`
   - "Regularny" gdy `total_spent >= 1000`
   - "Okazjonalny" w pozostalych przypadkach

   Zapisz wynik do zmiennej `customer_segments`.

d) Policz ilu klientow nalezy do kazdego segmentu. Zapisz do zmiennej `segment_counts` (DataFrame z kolumnami `segment` i `liczba_klientow`).

e) Zapisz liczbe segmentow do zmiennej `num_segments` (powinna wynosic 3).

**Wskazowka:** `F.when(warunek, wartosc).when(warunek2, wartosc2).otherwise(wartosc_domyslna)`

In [ ]:
# Cwiczenie 7a: Odfiltruj niezalogowanych klientow
# TODO: uzyj filter() lub dropna() aby usunac wiersze z NULL w customer_id
df_logged = ...

print(f"Zamowienia zalogowanych klientow: {df_logged.count():,}")

# Cwiczenie 7b: Statystyki per klient
# TODO: groupBy customer_id i oblicz total_spent, order_count, avg_order_value
customer_stats = (
    df_logged.groupBy("customer_id")
    .agg(
        ...  # total_spent
        ...  # order_count
        ...  # avg_order_value
    )
)

# Cwiczenie 7c: Dodaj kolumne segment
# TODO: uzyj F.when().when().otherwise() do przypisania segmentu
customer_segments = customer_stats.withColumn(
    "segment",
    F.when(..., "VIP")
     .when(..., "Regularny")
     .otherwise("Okazjonalny")
)

# Cwiczenie 7d: Policz klientow per segment
# TODO: groupBy segment i zlicz klientow
segment_counts = (
    customer_segments.groupBy("segment")
    .agg(F.count("customer_id").alias("liczba_klientow"))
    .orderBy(F.desc("liczba_klientow"))
)
segment_counts.show()

# Cwiczenie 7e: Liczba segmentow
# TODO: policz ile jest roznych segmentow
num_segments = ...

print(f"Liczba segmentow: {num_segments}")

In [ ]:
# @title Wskazowka do Cwiczenia 7
print("""Wskazowki:
Cwiczenie 7a -- dwa rownowazne sposoby:
  df_logged = df.filter(F.col("customer_id").isNotNull())
  lub: df_logged = df.dropna(subset=["customer_id"])

Cwiczenie 7b:
  .agg(
      F.round(F.sum("order_value"), 2).alias("total_spent"),
      F.count("order_id").alias("order_count"),
      F.round(F.avg("order_value"), 2).alias("avg_order_value"),
  )

Cwiczenie 7c:
  F.when(F.col("total_spent") > 5000, "VIP")
   .when(F.col("total_spent") >= 1000, "Regularny")
   .otherwise("Okazjonalny")

Cwiczenie 7e:
  num_segments = segment_counts.count()
""")

In [ ]:
# @title Test automatyczny -- Cwiczenie 7
try:
    assert df_logged.count() > 470_000, (
        f"Zbyt malo wierszy po filtrowaniu: {df_logged.count():,}. "
        "Niezalogowani to ok. 5% zamowien, wiec powinno pozostac ~475 000."
    )
    assert customer_segments.columns.__contains__("segment"), (
        "Kolumna 'segment' nie zostala dodana do customer_segments."
    )
    assert num_segments == 3, (
        f"Oczekiwano 3 segmentow (VIP, Regularny, Okazjonalny), otrzymano {num_segments}."
    )
    total_segmented = segment_counts.agg(F.sum("liczba_klientow")).collect()[0][0]
    assert total_segmented > 0, "Brak klientow w segmentach."

    sc_pd = segment_counts.toPandas()
    print("POPRAWNIE. Segmentacja klientow poprawna.")
    print()
    for _, row in sc_pd.iterrows():
        pct = 100 * row["liczba_klientow"] / total_segmented
        print(f"  {row['segment']:12s}: {row['liczba_klientow']:>8,} klientow ({pct:.1f}%)")
    print()
    print("Wniosek biznesowy: Wiekszosc klientow to klienci okazjonalni lub regularni.")
    print("Segment VIP, mimo ze nieliczny, generuje nieproporcjonalnie wysoki przychod.")
    print("Program lojalnosciowy powinien priorytetowo traktowac utrzymanie klientow VIP")
    print("oraz konwersje segmentu Regularnego do VIP.")
except NameError as e:
    print(f"BLAD: Zmienna nie zostala zdefiniowana -- {e}")
except AssertionError as e:
    print(f"NIEPOPRAWNIE: {e}")

In [ ]:
# Wizualizacja wynikow Cwiczenia 7 -- uruchom po ukonczeniu cwiczenia

sc_pd = segment_counts.toPandas()
total_customers = sc_pd["liczba_klientow"].sum()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

segment_colors = {"VIP": "#c0392b", "Regularny": "#2980b9", "Okazjonalny": "#27ae60"}
colors = [segment_colors.get(s, "#7f8c8d") for s in sc_pd["segment"]]

axes[0].bar(sc_pd["segment"], sc_pd["liczba_klientow"], color=colors)
axes[0].set_ylabel("Liczba klientow")
axes[0].set_title("Liczba klientow per segment")
for i, (_, row) in enumerate(sc_pd.iterrows()):
    pct = 100 * row["liczba_klientow"] / total_customers
    axes[0].text(i, row["liczba_klientow"] + 200, f"{pct:.1f}%", ha="center", fontsize=10)

vip_spent = customer_segments.filter(F.col("segment") == "VIP").agg(F.sum("total_spent")).collect()[0][0] or 0
reg_spent = customer_segments.filter(F.col("segment") == "Regularny").agg(F.sum("total_spent")).collect()[0][0] or 0
occ_spent = customer_segments.filter(F.col("segment") == "Okazjonalny").agg(F.sum("total_spent")).collect()[0][0] or 0

segment_revenue = {"VIP": vip_spent, "Regularny": reg_spent, "Okazjonalny": occ_spent}
labels = list(segment_revenue.keys())
values = [segment_revenue[k] / 1e6 for k in labels]
rev_colors = [segment_colors.get(k, "#7f8c8d") for k in labels]

axes[1].bar(labels, values, color=rev_colors)
axes[1].set_ylabel("Laczny przychod (mln PLN)")
axes[1].set_title("Laczny przychod per segment")

plt.tight_layout()
plt.show()
print("Interpretacja: Wykres po lewej pokazuje liczebnosc segmentow.")
print("Wykres po prawej pokazuje ich wklad w calkowity przychod.")
print("Jesli VIP generuje duzy przychod mimo malej liczebnosci, to jest to potwierdzenie")
print("reguly Pareto: 20% klientow odpowiada za 80% przychodu.")

---

## Podsumowanie laboratorium

Gratulacje! Ukonczyłes laboratorium z analizy danych e-commerce przy uzyciu Apache Spark.

### Wnioski biznesowe zebrane podczas analizy

| Obszar | Wniosek |
|--------|---------|
| Eksploracja danych | Zbior zawiera 500 000 transakcji, z czego ok. 5% pochodzi od niezalogowanych klientow |
| Kategorie produktow | Elektronika generuje najwyzszy laczny przychod; Ksiazki i Kosmetyki maja najnizszy sredni koszyk |
| Regiony | Mazowieckie dominuje pod wzgledem sprzedazy; roznice miedzy regionami sa znaczne |
| Sezonowosc | Szczytowy miesiac sprzedazy to listopad lub grudzien (efekt Black Friday i swiat) |
| Metody platnosci | Zamowienia oplacone przy odbiorze (Pobranie) sa dostarczane srednio 1-2 dni dluzej |
| Zwroty | Odzież ma najwyzszy wskaznik zwrotow; nalezy wzmocnic opisy rozmiarowek i zdjecia 360 stopni |
| Segmentacja klientow | Klienci VIP stanowia mniejszosc, ale generuja nieproporcjonalnie wysoki przychod |

### Poznane metody PySpark

- `df.printSchema()`, `df.show()`, `df.describe()` — eksploracja danych
- `df.filter()`, `df.select()`, `df.withColumn()` — selekcja i transformacja
- `df.groupBy().agg()` — grupowanie i agregacje
- `F.sum()`, `F.avg()`, `F.count()`, `F.when()` — funkcje agregujace i warunkowe
- `df.createOrReplaceTempView()` + `spark.sql()` — Spark SQL
- `df.cache()` — cachowanie dla wielokrotnego uzycia
- `df.toPandas()` — eksport do pandas i wizualizacja matplotlib

### Co dalej?

- Sprawdz dokumentacje PySpark: https://spark.apache.org/docs/latest/api/python/
- Pobaw sie z wiekszymi zbiorami danych (NYC Taxi: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)
- Sprobuj tego samego cwiczenia na klastrze Dataproc w GCP — poczujesz rozne miedzy lokalnym Sparkiem a klastrem!
- Nastepny temat: Kafka Streaming — analiza danych w czasie rzeczywistym

In [ ]:
# Zatrzymanie SparkSession po zakonczeniu pracy
spark.stop()
print("SparkSession zatrzymany. Laboratorium zakonczone.")